In [79]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_predict


In [21]:
df = pd.read_csv(Path.cwd().parent.parent/ "data"/ "processed"/ "heart_disease_health_indicators_BRFSS2015.csv")

In [9]:
X = df.drop(columns=["HeartDiseaseorAttack"])
y = df["HeartDiseaseorAttack"]
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (295751, 21)
y shape: (295751,)

Target distribution:
HeartDiseaseorAttack
0.0    270728
1.0     25023
Name: count, dtype: int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (236600, 21)
X_test : (59151, 21)
y_train: (236600,)
y_test : (59151,)


# Test Models to Generalize Params:

### Model A:
- n_estimators=300
- max_depth=4
- learning_rate=0.05


In [38]:
model_a = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
model_a.fit(X_train, y_train)

y_pred_a = model_a.predict(X_test)
y_prob_a = model_a.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred_a)

print("Accuracy :", accuracy_score(y_test, y_pred_a))
print("Precision:", precision_score(y_test, y_pred_a))
print("Recall   :", recall_score(y_test, y_pred_a))
print("F1       :", f1_score(y_test, y_pred_a))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_a))
print(cm)

#test thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print("\nTHRESHOLD RANGE TEST:")
for threshold in thresholds:
    y_pred_custom = (y_prob_a >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_custom)
    recall = recall_score(y_test, y_pred_custom)
    f1 = f1_score(y_test, y_pred_custom)

    print(
        f"Threshold={threshold:.1f} | "
        f"Precision={precision:.3f} | "
        f"Recall={recall:.3f} | "
        f"F1={f1:.3f}"
    )

print("Min :", y_prob_a.min())
print("Max :", y_prob_a.max())
print("Mean:", y_prob_a.mean())

Accuracy : 0.9177359638890298
Precision: 0.5807200929152149
Recall   : 0.0999000999000999
F1       : 0.17047391749062393
ROC-AUC  : 0.8614019286959117
[[53785   361]
 [ 4505   500]]

THRESHOLD RANGE TEST:
Threshold=0.1 | Precision=0.247 | Recall=0.782 | F1=0.375
Threshold=0.2 | Precision=0.338 | Recall=0.559 | F1=0.421
Threshold=0.3 | Precision=0.415 | Recall=0.368 | F1=0.390
Threshold=0.4 | Precision=0.503 | Recall=0.207 | F1=0.293
Threshold=0.5 | Precision=0.581 | Recall=0.100 | F1=0.170
Threshold=0.6 | Precision=0.688 | Recall=0.049 | F1=0.091
Threshold=0.7 | Precision=0.694 | Recall=0.012 | F1=0.023
Threshold=0.8 | Precision=0.750 | Recall=0.001 | F1=0.001
Threshold=0.9 | Precision=0.000 | Recall=0.000 | F1=0.000
Min : 0.0007556155
Max : 0.84443945
Mean: 0.084161274


c:\Users\giaba\anaconda3\envs\vis_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Model B:
- n_estimators=300
- max_depth=4
- learning_rate=0.1

In [ ]:
model_b = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

model_b.fit(X_train, y_train)

y_pred_b = model_b.predict(X_test)
y_prob_b = model_b.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_b))
print("Precision:", precision_score(y_test, y_pred_b))
print("Recall   :", recall_score(y_test, y_pred_b))
print("F1       :", f1_score(y_test, y_pred_b))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_b))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_b))

threshold = 0.2

y_pred_b_02 = (y_prob_b >= threshold).astype(int)
print("\nTHRESHOLD TEST 0.2:")
print("Precision:", precision_score(y_test, y_pred_b_02))
print("Recall   :", recall_score(y_test, y_pred_b_02))
print("F1       :", f1_score(y_test, y_pred_b_02))

Accuracy : 0.9172964108806275
Precision: 0.5621562156215621
Recall   : 0.1020979020979021
F1       : 0.1728102806898884
ROC-AUC  : 0.8606351337134774

Confusion Matrix:
[[53748   398]
 [ 4494   511]]

 THRESHOLD TEST 0.2:
Precision: 0.3363537513591881
Recall   : 0.5562437562437562
F1       : 0.4192139737991266


### Model C:
- n_estimators=600
- max_depth=4
- learning_rate=0.05

In [40]:
model_c = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

model_c.fit(X_train, y_train)

y_prob_c = model_c.predict_proba(X_test)[:, 1]
y_pred_c = (y_prob_c >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred_c))
print("Precision:", precision_score(y_test, y_pred_c))
print("Recall   :", recall_score(y_test, y_pred_c))
print("F1       :", f1_score(y_test, y_pred_c))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_c))

Accuracy : 0.9174316579601359
Precision: 0.566410537870472
Recall   : 0.1030969030969031
F1       : 0.1744421906693712
ROC-AUC  : 0.8608320372421137


# Grid Search:

In [74]:
param_grid_7 = {
    "learning_rate": [0.28, 0.30, 0.32, 0.34],
    "max_depth": [9],
    "n_estimators": [350, 375, 400]
}

model = XGBClassifier(
    learning_rate=0.28,
    max_depth=9,
    n_estimators=350,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight = 5,
    reg_lambda=2,
    gamma=0.05,
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "reg_alpha": [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1]
}

grid_reg1 = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

grid_reg1.fit(X_train, y_train)

print(grid_reg1.best_params_)
print(grid_reg1.best_score_)
best_model = grid_reg1.best_estimator_

results_df = pd.DataFrame(grid_reg1.cv_results_).drop(columns=["params", "split0_test_score", "split1_test_score", "split2_test_score", "mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time"])
results_df = results_df.sort_values("rank_test_score")
print("\nGrid Search Results:")
print(results_df)

Fitting 3 folds for each of 7 candidates, totalling 21 fits
{'reg_alpha': 0}
0.23373973692328395

Grid Search Results:
   param_reg_alpha  mean_test_score  std_test_score  rank_test_score  \
0             0.00         0.233740        0.002522                1   
5             0.50         0.232050        0.006867                2   
1             0.01         0.231051        0.006554                3   
4             0.20         0.230676        0.006217                4   
6             1.00         0.230635        0.002097                5   
2             0.05         0.230533        0.006718                6   
3             0.10         0.229866        0.005181                7   

   split0_train_score  split1_train_score  split2_train_score  \
0            0.769343            0.770139            0.773199   
5            0.778515            0.770797            0.780176   
1            0.768678            0.771057            0.772892   
4            0.775363            0.768467   

In [66]:
# =========================
# 1. Predictions on TRAIN
# =========================
y_train_pred = best_model.predict(X_train)
y_train_proba = best_model.predict_proba(X_train)[:, 1]

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
train_auc = roc_auc_score(y_train, y_train_proba)


# =========================
# 2. Cross-validation F1
# =========================
cv_f1 = grid_search.best_score_


# =========================
# 3. Print results
# =========================
print("=== BEST MODEL ===")
print(best_model)

print("\n=== TRAIN PERFORMANCE ===")
print(f"Accuracy : {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall   : {train_recall:.4f}")
print(f"F1       : {train_f1:.4f}")
print(f"ROC-AUC  : {train_auc:.4f}")

print("\n=== CROSS-VALIDATION ===")
print(f"CV F1    : {cv_f1:.4f}")

print("\n=== TRAIN - CV GAP ===")
print(f"F1 gap   : {train_f1 - cv_f1:.4f}")

=== BEST MODEL ===
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.05,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.34, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

=== TRAIN PERFORMANCE ===
Accuracy : 0.9787
Precision: 0.9881
Recall   : 0.7573
F1       : 0.8574
ROC-AUC  : 0.9912

=== CROSS-VALIDATION ===
CV F1    : 0.2380

=== TRAIN - CV GAP ===
F1 gap   : 0.6194


In [75]:
best_model = XGBClassifier(
    learning_rate=0.28,
    max_depth=9,
    n_estimators=350,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.05,
    reg_alpha=0,
    reg_lambda=2,
    random_state=42,
    eval_metric="logloss"
)

cv_proba = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=3,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

cv_pred = (cv_proba >= 0.5).astype(int)

print(classification_report(y_train, cv_pred))
print(confusion_matrix(y_train, cv_pred))
print("ROC-AUC:", roc_auc_score(y_train, cv_proba))

              precision    recall  f1-score   support

         0.0       0.93      0.98      0.95    216582
         1.0       0.41      0.16      0.23     20018

    accuracy                           0.91    236600
   macro avg       0.67      0.57      0.59    236600
weighted avg       0.88      0.91      0.89    236600

[[211800   4782]
 [ 16736   3282]]
ROC-AUC: 0.8145549362666987


In [80]:
model_candidate = XGBClassifier(
    learning_rate=0.28,
    max_depth=9,
    n_estimators=350,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.05,
    reg_alpha=0,
    reg_lambda=2,
    random_state=42,
    eval_metric="logloss"
)
model_candidate.fit(X_train, y_train)
test_proba = model_candidate.predict_proba(X_test)[:, 1]

test_pred = (test_proba >= 0.15).astype(int)

In [81]:
print("=== FINAL TEST EVALUATION ===")

print("Accuracy :", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall   :", recall_score(y_test, test_pred))
print("F1       :", f1_score(y_test, test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, test_proba))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, test_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, test_pred))

=== FINAL TEST EVALUATION ===
Accuracy : 0.8430457642305287
Precision: 0.2813490035769034
Recall   : 0.55004995004995
F1       : 0.3722785665990534
ROC-AUC  : 0.8232169226998023

=== CONFUSION MATRIX ===
[[47114  7032]
 [ 2252  2753]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         0.0       0.95      0.87      0.91     54146
         1.0       0.28      0.55      0.37      5005

    accuracy                           0.84     59151
   macro avg       0.62      0.71      0.64     59151
weighted avg       0.90      0.84      0.86     59151



# Random Search:

In [ ]:
param_distributions = {
    "learning_rate": [0.08, 0.10, 0.12, 0.13, 0.15, 0.17, 0.20],
    "max_depth": [3, 4, 5, 6, 7],
    "n_estimators": [200, 250, 275, 300, 325, 350, 400],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.05, 0.1, 0.2],
    "reg_alpha": [0, 0.05, 0.1, 0.2],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0]
}

model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)
best_model = random_search.best_estimator_

results_df = pd.DataFrame(random_search.cv_results_).drop(columns=["params", "split0_test_score", "split1_test_score", "split2_test_score", "mean_fit_time", "std_fit_time", "mean_score_time", "std_score_time"])
results_df = results_df.sort_values("rank_test_score")
print("\nRandom Search Results:")
print(results_df)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
{'subsample': 0.9, 'reg_lambda': 2.0, 'reg_alpha': 0, 'n_estimators': 325, 'min_child_weight': 1, 'max_depth': 7, 'learning_rate': 0.17, 'gamma': 0.05, 'colsample_bytree': 0.8}
0.20153375340362167

Random Search Results:
    param_subsample  param_reg_lambda  param_reg_alpha  param_n_estimators  \
3               0.9               2.0             0.00                 325   
15              0.7               1.0             0.20                 275   
0               0.9               1.0             0.10                 250   
22              0.8               0.5             0.00                 250   
7               0.7               1.0             0.05                 275   
10              0.9               0.5             0.05                 400   
26              0.8               1.0             0.10                 275   
13              0.9               1.5             0.00                 275   
19              

In [70]:
candidates = [
    # Grid #4
    {
        "round": 4,
        "learning_rate": 0.20,
        "max_depth": 7,
        "n_estimators": 350,
        "cv_f1": 0.210244083400741
    },
    {
        "round": 4,
        "learning_rate": 0.20,
        "max_depth": 7,
        "n_estimators": 325,
        "cv_f1": 0.2091035160390191
    },
    {
        "round": 4,
        "learning_rate": 0.20,
        "max_depth": 7,
        "n_estimators": 300,
        "cv_f1": 0.2071990385914755
    },

    # Grid #5
    {
        "round": 5,
        "learning_rate": 0.24,
        "max_depth": 8,
        "n_estimators": 350,
        "cv_f1": 0.22074069158750412
    },
    {
        "round": 5,
        "learning_rate": 0.24,
        "max_depth": 8,
        "n_estimators": 325,
        "cv_f1": 0.22033167710236554
    },
    {
        "round": 5,
        "learning_rate": 0.24,
        "max_depth": 8,
        "n_estimators": 375,
        "cv_f1": 0.22011479930091907
    },

    # Grid #6
    {
        "round": 6,
        "learning_rate": 0.28,
        "max_depth": 9,
        "n_estimators": 375,
        "cv_f1": 0.23255328879501344
    },
    {
        "round": 6,
        "learning_rate": 0.28,
        "max_depth": 9,
        "n_estimators": 350,
        "cv_f1": 0.23189769883439929
    },
    {
        "round": 6,
        "learning_rate": 0.26,
        "max_depth": 9,
        "n_estimators": 350,
        "cv_f1": 0.2311773283664309
    }
]


results = []

for candidate in candidates:

    model = XGBClassifier(
        learning_rate=candidate["learning_rate"],
        max_depth=candidate["max_depth"],
        n_estimators=candidate["n_estimators"],

        # Fixed regularization from Random Search best model
        subsample=0.9,
        colsample_bytree=0.8,
        min_child_weight=1,
        gamma=0.05,
        reg_alpha=0,
        reg_lambda=2.0,

        random_state=42,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    # Training predictions
    y_train_pred = model.predict(X_train)
    y_train_proba = model.predict_proba(X_train)[:, 1]

    train_f1 = f1_score(y_train, y_train_pred)
    train_auc = roc_auc_score(y_train, y_train_proba)

    results.append({
        "round": candidate["round"],
        "learning_rate": candidate["learning_rate"],
        "max_depth": candidate["max_depth"],
        "n_estimators": candidate["n_estimators"],
        "train_f1": train_f1,
        "cv_f1": candidate["cv_f1"],
        "f1_gap": train_f1 - candidate["cv_f1"],
        "train_auc": train_auc
    })


comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by="cv_f1",
    ascending=False
)

print(comparison_df.to_string(index=False))

 round  learning_rate  max_depth  n_estimators  train_f1    cv_f1   f1_gap  train_auc
     6           0.28          9           375  0.814730 0.232553 0.582177   0.986144
     6           0.28          9           350  0.800711 0.231898 0.568813   0.984543
     6           0.26          9           350  0.779016 0.231177 0.547839   0.981662
     5           0.24          8           350  0.629843 0.220741 0.409103   0.960586
     5           0.24          8           325  0.610674 0.220332 0.390342   0.957739
     5           0.24          8           375  0.648199 0.220115 0.428084   0.963241
     4           0.20          7           350  0.446331 0.210244 0.236087   0.929887
     4           0.20          7           325  0.429430 0.209104 0.220326   0.927032
     4           0.20          7           300  0.414887 0.207199 0.207688   0.924159
